In [0]:
import urllib.request
import os
import shutil

#### EXTRACT `TAXI_ZONE_LOOKUP.csv` FROM API TO ADLS

In [0]:
url = f'https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv'
response = urllib.request.urlopen(url)

dir_path = f'/Volumes/nyctaxi/landing/lookup'
os.makedirs(dir_path, exist_ok = True)

local_file_path = '/Volumes/nyctaxi/landing/lookup/taxi_zone_lookup.csv'
with open(local_file_path, 'wb') as f:
    shutil.copyfileobj(response, f)

#### READ FROM VOLUME

In [0]:
taxi_lookup_df = (spark.read.format('csv')
                            .option('header', 'true')
                            .load('/Volumes/nyctaxi/landing/lookup/taxi_zone_lookup.csv')
                            )

In [0]:
from pyspark.sql.functions import col, cast, current_timestamp, lit
from pyspark.sql.types import IntegerType, TimestampType

taxi_lookup_trans_df = taxi_lookup_df.select(
                                            col('LocationID').cast(IntegerType()).alias('location_id'),
                                            col('Borough').alias('borough'),
                                            col('Zone').alias('zone'),
                                            col('service_zone'),
                                            current_timestamp().alias('effective_start_date'),
                                            lit(None).cast(TimestampType()).alias('effective_end_date')
                                            )

taxi_lookup_trans_df.limit(1).display()

#### LOAD LOOKUP INTO
- `NYCTAXI.BRONZE.TAXI_ZONE_LOOKUP`

In [0]:
taxi_lookup_trans_df.write.mode('overwrite').saveAsTable('nyctaxi.bronze.taxi_zone_lookup')

In [0]:
dbutils.notebook.exit('TAXI LOOKUP FILE HAS BEEN LOADED INTO NYCTAXI.BRONZE.TAXI_ZONE_LOOKUP')